In [ ]:
import pyproj
pyproj.datadir.set_data_dir("/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/proj/9.2.0/share/proj")

In [ ]:
import xarray as xr
from pathlib import Path
from ecod_calculation import cod_from_water_path
from utils_cartopy import plot_global_field, plot_north_pole_field

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
output_dir = Path('tcwv_validation_slides')
output_dir.mkdir(exist_ok=True)

path_kernels = Path('data/fal/kernels')

In [ ]:
tcwv_paths = path_kernels.glob("RRTM_kernel_monthly_*_tcwv_TOA_SFC.nc")
#ecod_paths = path_era5.glob("era5_true_tcc_ecod_monthly*.nc")

In [ ]:
ds_tcwv = xr.open_mfdataset(tcwv_paths)
#ds_ecod = xr.open_mfdataset(ecod_paths)

In [ ]:
print(ds_tcwv)

In [ ]:
year=2011

fig, ax = plt.subplots(3, 4, figsize=(16,8), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)), layout='constrained')

for i in range(12):
    r = i // 4
    c = i % 4
    date = f"{year}-{i+1:02d}"
    field = ds_tcwv.sel(date=date, all_clr='clr')
    contour = ax[r][c].contourf(
        field.longitude, field.latitude, field.tcwv[0], 
        levels=np.arange(0, 6.1, 0.25), 
        transform=ccrs.PlateCarree(), 
        cmap='Reds', 
        vmin=0, 
        vmax=6,
        extend='both'
    )
    ax[r][c].coastlines()
    ax[r][c].gridlines()
    ax[r][c].set_title(date)

fig.colorbar(contour, ax=ax,  label="W/m^2 K", orientation='horizontal', )

In [ ]:
ds_tcwv.sel(all_clr='clr', latitude=-90, date='2011-01').tcwv.compute()

In [ ]:
fig = plt.figure(figsize=(10, 6))

ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180)) # Try ccrs.PlateCarree() or ccrs.Mollweide()
field = ds_tcwv.mean('date').sel(all_clr='all')

contour = ax.contourf(
    field.longitude, field.latitude, field.tcwv, 
    levels=np.arange(-1, 1.05, 0.1), 
    transform=ccrs.PlateCarree(), 
    cmap='RdBu_r', 
    vmin=-1, 
    vmax=1
)

ax.coastlines()
ax.gridlines()
plt.colorbar(contour, orientation='horizontal', pad=0.05, label='W/m^2 K')
plt.title('mean TOA allsky SW wv kernel')

plt.show()